# Databricks Declarative Automation Bundles (DABs)
### Introduction to deployment, environments, and CI/CD

**Goal:** Understand what DABs solve, what goes inside a bundle, how deployment works, and how GitHub Actions fits into the workflow.

**Official documentation:** [What are Declarative Automation Bundles?](https://docs.databricks.com/aws/en/dev-tools/bundles)

> DABs were formerly called **Databricks Asset Bundles**. The current name is **Declarative Automation Bundles**.

## 1. What problem are we solving?

Imagine we create a Databricks Job and Pipeline in DEV. Without a standard deployment approach, teams may manually recreate them in UAT and PROD. Small differences can appear: a missing task, a different schedule, a wrong notebook path, or a different setting.

**This is configuration drift:** environments that were supposed to be identical gradually become different.

## 2. Without DABs: manual deployment
![Screenshot 2026-11-20 123913_1795757362694.png](./Screenshot 2026-11-20 123913_1795757362694.png "Screenshot 2026-11-20 123913_1795757362694.png")

## 3. With DABs: one project, multiple targets
We maintain one source-controlled project and define environment-specific values for DEV, UAT, and PROD. We deploy the same bundle definition to the selected target.

![Screenshot 2026-11-20 123931_1795159134212.png](./Screenshot 2026-11-20 123931_1795159134212.png "Screenshot 2026-11-20 123931_1795159134212.png")

## 4. What are DABs?

**Declarative Automation Bundles** let us define a Databricks project as code and deploy it as one unit.

A bundle can contain:
- **Source code:** notebooks, Python, SQL files.
- **Resource definitions:** Jobs, Lakeflow Declarative Pipelines, and other supported Databricks resources.
- **Configuration:** workspace, target, variables, permissions and deployment settings.
- **Tests:** unit and integration tests where applicable.

The key idea: describe the desired setup in files, validate it, then deploy it using the Databricks CLI.

## 5. Official Databricks architecture diagram

It shows the relationship between the local project bundle, version control, CI/CD, and Databricks workspaces.

![Official Databricks DAB CI/CD architecture](https://docs.databricks.com/aws/en/assets/images/bundles-cicd-53be5f4860e8ebcedc2702f870290cda.png)

Source: [Databricks — What are Declarative Automation Bundles?](https://docs.databricks.com/aws/en/dev-tools/bundles)

## 6. Our simplified end-to-end architecture
local development → GitHub → GitHub Actions → authentication → target workspace.
![Screenshot 2026-11-20 123734_1795159248763.png](./Screenshot 2026-11-20 123734_1795159248763.png "Screenshot 2026-11-20 123734_1795159248763.png")

## 7. What does each part do?

| Component | Purpose |
|---|---|
| Developer environment | Write and test Python/SQL/YAML code |
| GitHub repository | Store code and track changes/version history |
| Pull Request (PR) | Review and approve proposed changes |
| GitHub Actions | Automatically run checks and deployment steps |
| Authentication | Give the workflow authorized access to Databricks |
| Target workspace | Where the bundle resources are deployed |

**Deployment** means applying the approved project configuration and files to the target workspace so they are available for actual use.

## 8. Typical bundle project structure

```text
sales_project/
├── databricks.yml
├── resources/
│   ├── sales_job.yml
│   └── sales_pipeline.yml
├── src/
│   ├── ingest.py
│   └── transform.py
└── tests/
    └── test_transform.py
```

- `databricks.yml`: main bundle file; declares the bundle and includes resource files/targets.
- `resources/`: definitions for Jobs and Pipelines.
- `src/`: business logic.
- `tests/`: automated checks.

The project can have additional files and folders depending on the use case.

## 9. The three YAML concepts to remember

**1. Main configuration — `databricks.yml`**
- Bundle name
- Included resource YAML files
- Variables and substitutions
- Deployment targets such as `dev`, `uat`, `prod`

**2. Job resource YAML**
- Job name and task definitions
- Task dependencies (`depends_on`)
- Schedule/trigger and parameters, when configured

**3. Pipeline resource YAML**
- Pipeline name and configuration
- Source/code location
- Catalog, target schema and other pipeline settings

If scheduling and dependencies are declared in YAML, they are deployed with the resource. They do not need to be recreated manually after each deployment.

## 10. One bundle, different deployment targets

The same project can be deployed to different workspaces by selecting a target:

```bash
databricks bundle validate -t dev
databricks bundle deploy -t dev

databricks bundle validate -t uat
databricks bundle deploy -t uat

databricks bundle validate -t prod
databricks bundle deploy -t prod
```

Each target can supply environment-specific values such as workspace host, catalog, schema, permissions, or resource naming.

**Important:** separate DEV/UAT/PROD workspaces are an organizational setup, not something DABs automatically create for you.

## 11. Essential DAB CLI lifecycle

```bash
databricks bundle init       # create a starter bundle
databricks bundle validate   # check configuration
databricks bundle plan       # preview changes 
databricks bundle deploy    # deploy bundle to target
databricks bundle run <key> # run a declared resource
databricks bundle summary   # view deployed resources
```

Use `databricks bundle destroy` carefully: it removes resources managed by that bundle deployment.

**Remember:** validate checks the bundle configuration; deploy applies it to the selected workspace; run starts a deployed resource.

## 12. How GitHub CI/CD and authentication fit in

A common workflow:

1. Developer pushes a change or opens a PR.
2. GitHub Actions runs tests and bundle validation.
3. After approval/merge, a deployment workflow runs.
4. The workflow authenticates to Databricks.
5. It runs `databricks bundle deploy -t <target>`.

**Authentication basics**
- **U2M (user-to-machine):** a human developer signs in interactively; useful for local development.
- **M2M (machine-to-machine):** automation authenticates as a service principal.
- **OIDC/workload identity federation:** can let GitHub Actions obtain short-lived credentials without storing a long-lived client secret, when the Databricks account/workspace setup supports it.

GitHub PR checks can pass even if deployment authentication is unavailable. In that case, code is merged in GitHub but workspace resources are not updated.

## 13. Key takeaways and official references

- DABs package source code, resource definitions, configuration and tests as one deployable project.
- YAML describes the desired Jobs, Pipelines, tasks, dependencies, schedules and environment settings.
- One bundle can be deployed to different targets; target-specific values keep environments distinct.
- GitHub manages source control and reviews; GitHub Actions automates checks and deployment commands.
- Deployment is the step that applies the bundle to the Databricks workspace.

**Official learning links**
1. [What are Declarative Automation Bundles?](https://docs.databricks.com/aws/en/dev-tools/bundles)
2. [Develop Declarative Automation Bundles — lifecycle](https://docs.databricks.com/aws/en/dev-tools/bundles/work-tasks)
3. [Bundle configuration reference](https://docs.databricks.com/aws/en/dev-tools/bundles/settings)
4. [CI/CD workflows on Databricks](https://docs.databricks.com/aws/en/dev-tools/ci-cd/flows)

**Next practical step:** create a small bundle with one Python notebook, one Job, and one Pipeline, then validate and deploy it to DEV.